# A tour of the chaotic_sound engine

This notebook uses the Python reference engine (`chaotic_sound`) to render sounds from chaotic
systems tuned to musical notes. Run it with the lab environment (for example, open it in
VS Code and pick the `lab/.venv` kernel). Every cell renders in a second or two.

Background: `docs/math/tuning.md` (how tuning works) and `docs/spec-v1.md` (what each box does).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display
from scipy.signal import welch

from chaotic_sound.catalog import load_default_catalog
from chaotic_sound.patch import Patch
from chaotic_sound.integrate import trajectory
from chaotic_sound.systems import get_system, list_systems
from chaotic_sound.wav import write_wav

FS = 48_000
catalog = load_default_catalog()
for name in list_systems():
    e = catalog[name]
    print(f"{name:8s} {e.param:>5s} ∈ [{e.p_min:.4g}, {e.p_max:.4g}]  default {e.p_default:g} (knob {e.u_default:.2f}), "
          f"heard: {e.default_observable}, tuned by {e.period_method}")

## 1. One box: Rössler tuned to A3

A dynamical box integrates the ODE with its clock sped up by k = P·f₀, so that one
characteristic period P of the attractor lasts exactly 1/f₀ seconds.

In [ ]:
p = Patch()
p.add_block("dynamical", "osc", system="rossler", note="A3")
p.add_block("output", "out", volume=1.0)
p.connect("osc.out", "out.in")
audio = p.render(seconds=4.0, sample_rate=FS)
display(Audio(audio.T, rate=FS))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5))
t = np.arange(int(0.03 * FS)) / FS
ax1.plot(1000 * t, audio[FS : FS + len(t), 0])
ax1.set(xlabel="time (ms)", title="30 ms of waveform (≈ 6.6 periods of 220 Hz)")
f, pxx = welch(audio[:, 0], fs=FS, nperseg=2**14)
ax2.semilogy(f, pxx)
ax2.axvline(220, color="r", ls="--", label="A3 = 220 Hz")
ax2.set(xlim=(0, 3000), xlabel="frequency (Hz)", title="spectrum")
ax2.legend()
plt.show()

## 2. Same timbre, any pitch

Tuning only changes the speed, not the shape of the trajectory, so the timbre is identical at
every note. Try other systems: `"lorenz"`, `"chua"`, `"thomas"` or `"duffing"`.

In [ ]:
def voice(system, note, seconds=2.0, **params):
    p = Patch()
    p.add_block("dynamical", "osc", system=system, note=note, **params)
    p.add_block("output", "out", volume=1.0)
    p.connect("osc.out", "out.in")
    return p.render(seconds=seconds, sample_rate=FS)

display(Audio(np.vstack([voice("chua", n) for n in ("A2", "A3", "A4")]).T, rate=FS))

## 3. Moving a parameter = forcing, and pitch lock

A modulator ramps Lorenz's ρ across its whole chaotic range in 10 s. With pitch lock on
(the default), the time scale follows P(ρ), so the note stays put while the timbre changes.
With it off, the pitch rises by almost an octave.

In [ ]:
def lorenz_sweep(pitch_lock):
    p = Patch()
    p.add_block("modulator", "ramp", shape="ramp_up", period_s=10.0, mode="once")
    p.add_block("dynamical", "lorenz", system="lorenz", note="A3", pitch_lock=pitch_lock)
    p.add_block("output", "out", volume=1.0)
    p.connect("ramp.out", "lorenz.param")
    p.connect("lorenz.out", "out.in")
    return p.render(seconds=10.0, sample_rate=FS)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
for ax, lock in zip(axes, (True, False)):
    a = lorenz_sweep(lock)
    display(Audio(a.T, rate=FS))
    ax.specgram(a[:, 0], NFFT=4096, Fs=FS, noverlap=3072, cmap="magma", vmin=-140)
    ax.set(yscale="log", ylim=(50, 5000), xlabel="time (s)", title=f"pitch lock {'ON' if lock else 'OFF'}")
plt.show()

## 4. Your first patch idea

Two modulators, one on a Rössler parameter and one on a Lorenz amplitude, mixed in a Sum
box and sent to the Output. The patch can be saved as JSON; the same format will drive the
browser app.

In [ ]:
p = Patch()
p.add_block("modulator", "lfo_param", shape="sine", period_s=8.0)
p.add_block("modulator", "lfo_amp", shape="triangle", period_s=4.0)
p.add_block("dynamical", "rossler", system="rossler", note="A3")
p.add_block("dynamical", "lorenz", system="lorenz", note="E3")
p.add_block("sum", "mix", gain=0.8)
p.add_block("output", "out", volume=0.9)
p.connect("lfo_param.out", "rossler.param")
p.connect("lfo_amp.out", "lorenz.amp")
p.connect("rossler.out", "mix.in")
p.connect("lorenz.out", "mix.in")
p.connect("mix.out", "out.in")
audio = p.render(seconds=16.0, sample_rate=FS)
display(Audio(audio.T, rate=FS))
print(p.to_json()[:400], "...")
# write_wav("outputs/my_patch.wav", audio, FS)   # uncomment to export

## 5. Under the hood: attractor and tuning tables

On the left, the attractor whose motion you are hearing. On the right, the catalog's tuning
curve P(u), the characteristic period across the knob's range, which pitch lock follows.

In [ ]:
name = "chua"   # try any system
system, e = get_system(name), catalog[name]
traj, _ = trajectory(system, dt=e.h_max * 8, n_samples=60_000, h_max=e.h_max)
fig = plt.figure(figsize=(12, 4.5))
ax = fig.add_subplot(1, 2, 1, projection="3d")
ax.plot(traj[:, 0], traj[:, 1], traj[:, 2], lw=0.2)
ax.set(xlabel=system.variables[0], ylabel=system.variables[1], zlabel=system.variables[2], title=system.display_name)
ax2 = fig.add_subplot(1, 2, 2)
ax2.plot(e.param_value(e.u_grid), e.period_table, ".-")
ax2.axvline(e.p_default, color="k", ls="--", label="dysts default")
ax2.set(xlabel=e.param, ylabel="P (system time)", title=f"tuning curve ({e.period_method} on {e.period_observable})")
ax2.legend()
plt.show()

## Ideas to try

- Put two identical boxes (same system and note) into a Sum box and listen as chaos pulls
  them apart (`examples/patches/07_twins.json`).
- Use a square modulator on Chua's parameter to jump between its single and double scroll.
- Listen to Lorenz with `observable="x"`: same tuning, but you hear the random switching
  between the two lobes.
- Load any example patch with `Patch.load("../examples/patches/06_chaotic_chord.json")`.